# Ingestão Bronze

In [0]:
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

## Mapeamento e leitura dos arquivos csv

In [0]:
from pyspark.sql.functions import current_timestamp

path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_financials= f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_info= f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_metrics= f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_reviews= f"{landing_path}/movies_reviews.csv"

df_credits_and_tags = spark.read.csv(path_credits_and_tags, header= True, inferSchema= True)
df_financials = spark.read.csv(path_financials, header= True, inferSchema= True)
df_info = spark.read.csv(path_info, header= True, inferSchema= True)
df_metrics = spark.read.csv(path_metrics, header= True, inferSchema= True)
df_reviews = spark.read.csv(path_reviews, header= True, inferSchema= True)


## Impotando e fazendo a requisição da api

In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import current_timestamp

# Período de extração: últimos 30 dias
data_fim = datetime.now()
data_inicio = data_fim - timedelta(days=30)

# A API do Banco Central espera o formato MM-DD-AAAA
data_inicio_formatada = data_inicio.strftime("%m-%d-%Y")
data_fim_formatada = data_fim.strftime("%m-%d-%Y")

# Monta o endpoint da API PTAX
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# Faz a requisição à API
response = requests.get(url)
response.raise_for_status()
dados = response.json()
cotacoes = dados.get("value", [])

print(f"Período: {data_inicio_formatada} a {data_fim_formatada}")
print(f"Total de cotações obtidas: {len(cotacoes)}")

# Define o schema explicitamente para garantir compatibilidade com Spark Connect
schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True)
])

# Cria o DataFrame Spark a partir das cotações
rows = [Row(dataHoraCotacao=str(c["dataHoraCotacao"]), cotacaoCompra=float(c["cotacaoCompra"])) for c in cotacoes]
df_cotacao = spark.createDataFrame(rows, schema)

Período: 09-07-2026 a 10-07-2026
Total de cotações obtidas: 22


dataHoraCotacao,cotacaoCompra,ingestion_timestamp
2026-09-08 13:02:38.447299,5.085,2026-10-07T18:49:22.466Z
2026-09-09 13:11:35.346979,5.0973,2026-10-07T18:49:22.466Z
2026-09-10 13:09:28.70012,5.1143,2026-10-07T18:49:22.466Z
2026-09-11 13:07:22.532196,5.0912,2026-10-07T18:49:22.466Z
2026-09-14 13:10:08.144425,5.169,2026-10-07T18:49:22.466Z


## Gravação com adição do timestamp no momento da escrita

In [0]:
df_credits_and_tags \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.credits_and_tags")

df_financials \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.financials")

df_info \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.info")

df_metrics \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.metrics")

df_reviews \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.reviews")

df_cotacao \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")


display(spark.table(f"{catalog}.bronze.tb_cotacao_dolar").limit(5))

dataHoraCotacao,cotacaoCompra,ingestion_timestamp
2026-09-08 13:02:38.447299,5.085,2026-10-07T18:51:44.115Z
2026-09-09 13:11:35.346979,5.0973,2026-10-07T18:51:44.115Z
2026-09-10 13:09:28.70012,5.1143,2026-10-07T18:51:44.115Z
2026-09-11 13:07:22.532196,5.0912,2026-10-07T18:51:44.115Z
2026-09-14 13:10:08.144425,5.169,2026-10-07T18:51:44.115Z
